#### varaibles
- real gdp per capita growth - trend + cycle of real values? diff og gdp real per capita
- inflation = inflation trend + measurement errors???
- short rate = inflation trend + real rate trend 
- long rate = inflation trend + real rate trend + term trend
- inflation expectations??
- growth expextations??
- 

In [1]:
import Pkg
Pkg.activate("../")

using Revise

includet("../src/TCVAR/TCVAR.jl")

  Activating project at `~/projects/MacroFinanceScenarios`


In [2]:
using .TCVAR
using DataFrames, XLSX, TimeSeries
using StatsBase
using LinearAlgebra
using Plots


In [3]:
df = DataFrame(XLSX.readtable("../data/DelNegro.xlsx", "Sheet1"))
data_source = TimeArray(df; timestamp = :Date)

s   = timestamp(data_source[:BILL])
vals = values(data_source[:BILL])

# widen so the array can hold `missing`
new_vals = convert(Array{Union{Missing, eltype(vals)}}, vals)

new_vals[s .>= Date(2008, 10,1), :] .= missing

TBILL = TimeArray(s, new_vals, [:BILL])

data = merge(data_source[:Pi, :EPi, :EBILL, :TBlong], TBILL)

presample, data = data[Date(1954,01,01):Date(1959,12,31)], data[Date(1960,01,01):Date(2016,12,31)]

term = presample[:TBlong] .- presample[:BILL]
real_rate = presample[:BILL] .- presample[:Pi]

data =convert(Matrix{Union{Missing, Float64}},values(data))

display(presample)
display(data)  


24×5 TimeArray{Union{Missing, Float64}, 2, Date, Matrix{Union{Missing, Float64}}} 1954-01-01 to 1959-10-01
┌────────────┬──────────┬─────────┬─────────┬────────┬──────┐
│            │ Pi       │ EPi     │ EBILL   │ TBlong │ BILL │
├────────────┼──────────┼─────────┼─────────┼────────┼──────┤
│ 1954-01-01 │  2.01108 │ missing │ missing │   2.71 │ 1.06 │
│ 1954-04-01 │ -0.55724 │ missing │ missing │   2.63 │ 0.79 │
│ 1954-07-01 │ -1.31896 │ missing │ missing │   2.58 │ 0.88 │
│ 1954-10-01 │ -0.38172 │ missing │ missing │   2.64 │ 1.02 │
│ 1955-01-01 │   1.3246 │ missing │ missing │   2.81 │ 1.22 │
│ 1955-04-01 │  0.40624 │ missing │ missing │   2.86 │ 1.48 │
│ 1955-07-01 │  1.59788 │ missing │ missing │   2.98 │ 1.86 │
│ 1955-10-01 │  1.11152 │ missing │ missing │   2.95 │ 2.34 │
│ 1956-01-01 │  1.63748 │ missing │ missing │   2.95 │ 2.33 │
│ 1956-04-01 │  2.70964 │ missing │ missing │   3.07 │ 2.57 │
│ 1956-07-01 │  3.91252 │ missing │ missing │   3.19 │ 2.58 │
│     ⋮      │    ⋮     │

228×5 Matrix{Union{Missing, Float64}}:
  0.52872  1.6827   missing  4.28  3.87
  2.11204  1.6827   missing  4.16  2.99
  1.53004  1.6827   missing  3.87  2.36
  1.77444  1.6827   missing  3.93  2.31
  0.7474   1.6827   missing  3.85  2.35
 -0.0452   1.6827   missing  3.81  2.3
  1.4696   1.6827   missing  4.0   2.3
  0.45052  1.6827   missing  4.03  2.46
  1.77756  1.6827   missing  4.09  2.72
  1.38888  1.6827   missing  3.94  2.72
  1.04924  1.6827   missing  4.02  2.84
  1.18008  1.6827   missing  3.93  2.81
  1.1322   1.6827   missing  3.96  2.91
  ⋮                                
  1.96876  2.0     2.5       3.42   missing
  1.88544  2.0      missing  3.18   missing
  1.07024  2.0      missing  3.01   missing
 -0.01096  2.0      missing  2.69   missing
 -1.6194   2.0     2.6686    2.32   missing
  1.80952  1.975    missing  2.62   missing
  1.14     2.0      missing  2.65   missing
  0.39348  1.9      missing  2.6    missing
  0.28028  1.973   2.5       2.32   missing
  2.00056  

In [4]:
var.([values(presample[:Pi]), values(term), values(real_rate)])

3-element Vector{Float64}:
 2.2741804700492754
 0.4066775362318842
 1.595023391788406

In [5]:
mean.([values(presample[:Pi]), values(term), values(real_rate)])

3-element Vector{Float64}:
 1.706988333333333
 1.0258333333333336
 0.5717616666666668

In [6]:
presample_mean = mean(presample)
presample_mean = round.(presample_mean, digits=2)
display("presample mean")
display(presample_mean)

presample_variance = var(presample)
presample_variance = round.(presample_variance, digits=2)
display("presample variance")
display(presample_variance)
display(presample_variance .^ .5) 

display(var(term))

"presample mean"

1×5 TimeArray{Union{Missing, Float64}, 2, Date, Matrix{Union{Missing, Float64}}} 1959-10-01 to 1959-10-01
┌────────────┬──────┬─────────┬─────────┬────────┬──────┐
│            │ Pi   │ EPi     │ EBILL   │ TBlong │ BILL │
├────────────┼──────┼─────────┼─────────┼────────┼──────┤
│ 1959-10-01 │ 1.71 │ missing │ missing │    3.3 │ 2.28 │
└────────────┴──────┴─────────┴─────────┴────────┴──────┘

"presample variance"

1×5 TimeArray{Union{Missing, Float64}, 2, Date, Matrix{Union{Missing, Float64}}} 1959-10-01 to 1959-10-01
┌────────────┬──────┬─────────┬─────────┬────────┬──────┐
│            │ Pi   │ EPi     │ EBILL   │ TBlong │ BILL │
├────────────┼──────┼─────────┼─────────┼────────┼──────┤
│ 1959-10-01 │ 2.27 │ missing │ missing │   0.26 │ 0.97 │
└────────────┴──────┴─────────┴─────────┴────────┴──────┘

1×5 TimeArray{Union{Missing, Float64}, 2, Date, Matrix{Union{Missing, Float64}}} 1959-10-01 to 1959-10-01
┌────────────┬─────────┬─────────┬─────────┬──────────┬──────────┐
│            │ Pi      │ EPi     │ EBILL   │ TBlong   │ BILL     │
├────────────┼─────────┼─────────┼─────────┼──────────┼──────────┤
│ 1959-10-01 │ 1.50665 │ missing │ missing │ 0.509902 │ 0.984886 │
└────────────┴─────────┴─────────┴─────────┴──────────┴──────────┘

1×1 TimeArray{Float64, 1, Date, Vector{Float64}} 1959-10-01 to 1959-10-01
┌────────────┬─────────────┐
│            │ TBlong_BILL │
├────────────┼─────────────┤
│ 1959-10-01 │    0.406678 │
└────────────┴─────────────┘

In [7]:
n = 5 #number of observatin variables
nt = 3 #number of trends

priors = (
        initial_trend_mean = [2., .5, 1.],
        initial_cycle_mean = zeros(n),
        initial_trend_covariance = diagm(fill(1,nt)),
        trend_covariance_df = 100,
        trend_covariance_mean = diagm([2., 1., 1.] ./ 400),
        cycle_coeff_mean = zeros(n, n),
        cycle_coeff_shrinkage_param = .2,
        cycle_covariance_mean = diagm([2., .5, .5, 1., 1.]),  
        cycle_covariance_df = n+2
        )


(initial_trend_mean = [2.0, 0.5, 1.0], initial_cycle_mean = [0.0, 0.0, 0.0, 0.0, 0.0], initial_trend_covariance = [1 0 0; 0 1 0; 0 0 1], trend_covariance_df = 100, trend_covariance_mean = [0.005 0.0 0.0; 0.0 0.0025 0.0; 0.0 0.0 0.0025], cycle_coeff_mean = [0.0 0.0 … 0.0 0.0; 0.0 0.0 … 0.0 0.0; … ; 0.0 0.0 … 0.0 0.0; 0.0 0.0 … 0.0 0.0], cycle_coeff_shrinkage_param = 0.2, cycle_covariance_mean = [2.0 0.0 … 0.0 0.0; 0.0 0.5 … 0.0 0.0; … ; 0.0 0.0 … 1.0 0.0; 0.0 0.0 … 0.0 1.0], cycle_covariance_df = 7)

In [8]:
priors.cycle_covariance_mean

5×5 Matrix{Float64}:
 2.0  0.0  0.0  0.0  0.0
 0.0  0.5  0.0  0.0  0.0
 0.0  0.0  0.5  0.0  0.0
 0.0  0.0  0.0  1.0  0.0
 0.0  0.0  0.0  0.0  1.0

In [10]:
observation_trend_mapping  = [1 0 0  
                             1 0 0  
                             1 1 0  
                             1 1 1  
                             1 1 0 ]



trend_states_samples, cycle_states_samples, trend_covariance_samples, betas_samples, sigmas_samples = TCVAR.gibbs_sampler(data, observation_trend_mapping, priors; burnin = 50_000, n_samples = 50_000, thin=25, logging=false)

trend_states_mean, trend_states_lower, trend_states_upper = TCVAR.compute_posterior_statistics(trend_states_samples, credible_level=0.68)  
cycle_states_mean, cycle_states_lower, cycle_states_upper = TCVAR.compute_posterior_statistics(cycle_states_samples, credible_level=0.95) 

LoadError: ArgumentError: invalid argument #4 to LAPACK call

In [ ]:
show(err)

1-element ExceptionStack:
LoadError: SingularException(3)
Stacktrace:
  [1] checknonsingular
    @ ~/.julia/juliaup/julia-1.12.5+0.x64.linux.gnu/share/julia/stdlib/v1.12/LinearAlgebra/src/factorization.jl:69 [inlined]
  [2] _check_lu_success
    @ ~/.julia/juliaup/julia-1.12.5+0.x64.linux.gnu/share/julia/stdlib/v1.12/LinearAlgebra/src/lu.jl:84 [inlined]
  [3] #lu!#180
    @ ~/.julia/juliaup/julia-1.12.5+0.x64.linux.gnu/share/julia/stdlib/v1.12/LinearAlgebra/src/lu.jl:92 [inlined]
  [4] lu!
    @ ~/.julia/juliaup/julia-1.12.5+0.x64.linux.gnu/share/julia/stdlib/v1.12/LinearAlgebra/src/lu.jl:90 [inlined]
  [5] lu!(A::Hermitian{Float64, Matrix{Float64}}, pivot::RowMaximum; check::Bool, allowsingular::Bool)
    @ LinearAlgebra ~/.julia/juliaup/julia-1.12.5+0.x64.linux.gnu/share/julia/stdlib/v1.12/LinearAlgebra/src/lu.jl:103
  [6] lu! (repeats 2 times)
    @ ~/.julia/juliaup/julia-1.12.5+0.x64.linux.gnu/share/julia/stdlib/v1.12/LinearAlgebra/src/lu.jl:95 [inlined]
  [7] _lu
    @ ~/.julia/ju

In [ ]:
priors.initial_trend_covariance

3×3 Matrix{Int64}:
 1  0  0
 0  1  0
 0  0  1

In [ ]:
n_variables, n_trends = size(observation_trend_mapping)
initial_state_covariance = [priors.initial_trend_covariance zeros(n_trends, n_variables)
                                 zeros(n_variables, n_trends) priors.cycle_covariance_mean]

8×8 Matrix{Float64}:
 1.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0
 0.0  1.0  0.0  0.0  0.0  0.0  0.0  0.0
 0.0  0.0  1.0  0.0  0.0  0.0  0.0  0.0
 0.0  0.0  0.0  2.0  0.0  0.0  0.0  0.0
 0.0  0.0  0.0  0.0  0.5  0.0  0.0  0.0
 0.0  0.0  0.0  0.0  0.0  0.5  0.0  0.0
 0.0  0.0  0.0  0.0  0.0  0.0  1.0  0.0
 0.0  0.0  0.0  0.0  0.0  0.0  0.0  1.0

In [ ]:
st = 1
plot(trend_states_mean[:,st], color="green" )
plot!(trend_states_lower[:,st], color="blue")
plot!(trend_states_upper[:,st], color="blue")

UndefVarError: UndefVarError: `trend_states_mean` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

In [ ]:
st = 2
plot(trend_states_mean[:,st], color="green" )
plot!(trend_states_lower[:,st], color="blue")
plot!(trend_states_upper[:,st], color="blue")

UndefVarError: UndefVarError: `trend_states_mean` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

In [ ]:
st = 3
plot(trend_states_mean[:,st], color="green" )
plot!(trend_states_lower[:,st], color="blue")
plot!(trend_states_upper[:,st], color="blue")

UndefVarError: UndefVarError: `trend_states_mean` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

In [ ]:
st = 4
plot(trend_states_mean[:,st], color="green" )
plot!(trend_states_lower[:,st], color="blue")
plot!(trend_states_upper[:,st], color="blue")

UndefVarError: UndefVarError: `trend_states_mean` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

In [ ]:
st=1
plot(cycle_states_mean[:,st], color="green" )

UndefVarError: UndefVarError: `cycle_states_mean` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

In [ ]:
st=2
plot(cycle_states_mean[:,st], color="green" )

UndefVarError: UndefVarError: `cycle_states_mean` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

In [ ]:
st=3
plot(cycle_states_mean[:,st], color="green" )

UndefVarError: UndefVarError: `cycle_states_mean` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

In [ ]:
st=4
plot(cycle_states_mean[:,st], color="green" )

UndefVarError: UndefVarError: `cycle_states_mean` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

In [ ]:
summarystats(trend_covariance_samples)

UndefVarError: UndefVarError: `trend_covariance_samples` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

In [ ]:
summarystats(betas_samples)

UndefVarError: UndefVarError: `betas_samples` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

In [ ]:
summarystats(sigmas_samples)

UndefVarError: UndefVarError: `sigmas_samples` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

In [ ]:
plot(betas_samples)

UndefVarError: UndefVarError: `betas_samples` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

In [ ]:
Σc = mean(sigmas_samples).nt.mean
Σc = reshape(Σc, n, n)
display(Σc)

β = mean(betas_samples).nt.mean
β = reshape(β, n, n*1)
display(β)

Στ = mean(trend_covariance_samples).nt.mean
Στ = reshape(Στ, 4, 4)
display(cov2cor(Στ))
display(diag(Στ) .^ .5)

UndefVarError: UndefVarError: `sigmas_samples` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

In [ ]:
display(diag(Σc) .^ .5)

display(diag(Στ) .^ .5)

UndefVarError: UndefVarError: `Σc` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

In [ ]:
model = tc_var(observation_tend_mapping, β, Στ, Σc, priors.initial_trend_mean, priors.initial_cycle_mean, priors.initial_trend_covariance, priors.cycle_covariance_mean)

initial_states = [trend_states_mean[end,:]; cycle_states_mean[end,:]]

n_samples = 2_000
T = 100
states = zeros(n_samples, T, 8)

observations = zeros(n_samples, T, n)

for s in 1:2_000
    states[s, :, :], observations[s, :, :] = sample(model, initial_states, T)
end


UndefVarError: UndefVarError: `observation_tend_mapping` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

In [ ]:
transformed_scenarios = permutedims(observations, (3, 2, 1))[[1,2],:,:] ./400
periods = [1, 5, 10, 25]
freq = 4
assets_names = ["GDP", "CPI"]
ret_in_years = cum_returns_in_periods(transformed_scenarios, periods, freq, true)
print_scenarios_summary(ret_in_years, assets_names, string.(periods))

for a in 1:2
    print_scenarios_percentiles(ret_in_years[a, :, :], [.01, 0.025, .05, .25, .5, .75, .95, .975, .99], string.(periods), string.(assets_names[a]))
end  

UndefVarError: UndefVarError: `observations` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

In [ ]:
transformed_scenarios = permutedims(observations[:,:,[3,4]] , (3, 2, 1))
freq = 4

transformed_scenarios = transformed_scenarios[:,freq:freq:end,:]

periods = [1, 5, 10, 25]


assets_names = ["ShortRate", "LongRate"]
ret_in_years = transformed_scenarios = transformed_scenarios[:,periods,:]
print_scenarios_summary(ret_in_years, assets_names, string.(periods))

for a in 1:2
    print_scenarios_percentiles(ret_in_years[a, :, :], [.01, 0.025, .05, .25, .5, .75, .95, .975, .99], string.(periods), string.(assets_names[a]))
end  

UndefVarError: UndefVarError: `observations` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

In [ ]:
n_observations = size(model.T,1)
n_trends = 4
n_time_steps = size(data,1)
n_draws = 1000
trends_states = zeros(n_draws, n_time_steps, n_trends)

for s in 1:n_draws
    state_smoothed_samples = carter_kohn_sampler(model, values(data))
    trends_states[s,:,:] = state_smoothed_samples[:, 1:n_trends]
end

trend_states_mean, trend_states_lower, trend_states_upper = TCVAR.compute_posterior_statistics(trend_states_samples, credible_level=0.95) 

UndefVarError: UndefVarError: `model` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

In [ ]:
st = 2
plot(trend_states_mean[:,st], color="green" )
plot!(trend_states_lower[:,st], color="blue")
plot!(trend_states_upper[:,st], color="blue")

UndefVarError: UndefVarError: `trend_states_mean` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

In [ ]:
st = 3
plot(trend_states_mean[:,st], color="green" )
plot!(trend_states_lower[:,st], color="blue")
plot!(trend_states_upper[:,st], color="blue")

UndefVarError: UndefVarError: `trend_states_mean` not defined in `Main`
Suggestion: check for spelling errors or missing imports.